<a href="https://colab.research.google.com/github/kcymae/Computational-TCell-Epitope-Analysis/blob/main/1.%20Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **IEDB Database**

The [*IEDB Database*](https://www.iedb.org/) established in 2004 as a free resource to house experimental data related to adaptive immune epitopes and to also provide leading-edge epitope analysis and prediction tools

## **Install Libraries**




In [ ]:
pip install iedb

In [ ]:
import iedb

mhci_res = iedb.query_mhci_binding(method="recommended", sequence="ARFTGIKTA", allele="HLA-A*02:01", length=8)

mhcii_res = iedb.query_mhcii_binding(method="nn_align", sequence="SLYNTVATLYCVHQRIDV", allele="HLA-DRB1*01:01", length=None)

tcell_res = iedb.query_tcell_epitope(method="smm", sequence="SLYNTVATLYCVHQRIDV", allele="HLA-A*01:01", length=9, proteasome='immuno')

pep_res = iedb.query_peptide_prediction(method="mhcnp", sequence="SLYNTVATLYCVHQRIDV", allele="HLA-A*02:01", length=9)

bcell_res = iedb.query_bcell_epitope(method="Emini", sequence="VLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRFKHLKTE", window_size=9)


## **Selection of Biological Target Protein**

A viral antigenic protein was selected as the target for epitope prediction. The spike glycoprotein of `SARS-CoV-2` was chosen due to its immunogenic relevance and availability of sequence data

The protein sequence of the SARS-CoV-2 spike glycoprotein (Accession ID: P0DTC2, Gene: S) was retrieved in FASTA format from the UniProt database, representing the Wuhan-Hu-1 isolate (the reference proteome)



In [ ]:
from urllib.request import urlretrieve

genome_url = 'https://raw.githubusercontent.com/kcymae/Project-/refs/heads/main/P0DTC2_SPIKE_SARS2%20Spike%20glycopr.fasta'

genome_file_name = 'P0DTC2_SPIKE_SARS2 Spike glycopr.fasta'

urlretrieve(genome_url, genome_file_name)


('P0DTC2_SPIKE_SARS2 Spike glycopr.fasta',
 <http.client.HTTPMessage at 0x782c5af740e0>)

In [ ]:
infile = open('P0DTC2_SPIKE_SARS2 Spike glycopr.fasta')
for line in infile:
    print(line)

>sp|P0DTC2|SPIKE_SARS2 Spike glycoprotein OS=Severe acute respiratory syndrome coronavirus 2 OX=2697049 GN=S PE=1 SV=1

MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFS

NVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIV

NNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLE

GKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQT

LLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETK

CTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISN

CVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIAD

YNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPC

NGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVN

FNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITP

GTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSY

ECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTI

SVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQE

VFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDC

LGDIAARDLICA

## **Generation of Candidate Peptides**



In [ ]:
! pip install biopython
from Bio import SeqIO

Read the FASTA file

In [ ]:
fasta_file ="P0DTC2_SPIKE_SARS2 Spike glycopr.fasta"
record = SeqIO.read(fasta_file, "fasta")
protein_sequence = str(record.seq)

print("Sequence length:", len(protein_sequence))
print(protein_sequence[:50])

Sequence length: 1273
MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHS


Generate overlapping peptides

In [ ]:
def generate_peptides(sequence, length=9):
    peptides = []
    for i in range(len(sequence) - length + 1):
        peptide = sequence[i:i+length]
        peptides.append(peptide)
    return peptides

peptides = generate_peptides(protein_sequence, length=9)

print("Total peptides generated:", len(peptides))
print(peptides[:10])

Total peptides generated: 1265
['MFVFLVLLP', 'FVFLVLLPL', 'VFLVLLPLV', 'FLVLLPLVS', 'LVLLPLVSS', 'VLLPLVSSQ', 'LLPLVSSQC', 'LPLVSSQCV', 'PLVSSQCVN', 'LVSSQCVNL']


## **Prediction of MHC Binding Affinity**

Prediction of MHC Class I Binding Affinity Using IEDB

In [ ]:
import pandas as pd
import time

all_results = []

for i, pep in enumerate(peptides[:50]):
    try:
        if len(pep) != 9:
            continue

        # Query the API
        res = iedb.query_mhci_binding(
            method="recommended",
            sequence=pep,
            allele="HLA-A*02:01",
            length=9
        )

        # The response IS a DataFrame, so work with it directly
        if isinstance(res, pd.DataFrame):
            print(f"Peptide {i} ({pep}): {len(res)} results")

            # Add peptide column to the response
            res['peptide'] = pep

            # Append to all_results
            all_results.append(res)

        time.sleep(0.5)

    except Exception as e:
        print(f"Error processing peptide {i}: {str(e)}")
        continue

print(f"\n✅ Processing complete!")
print(f"Total peptides with results: {len(all_results)}")

# Combine all DataFrames into one
if all_results:
    df = pd.concat(all_results, ignore_index=True)
    print(f"\n📊 DataFrame shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print("\nFirst 10 rows:")
    print(df.head(10))
else:
    print("❌ No results collected")

Peptide 0 (MFVFLVLLP): 1 results
Peptide 1 (FVFLVLLPL): 1 results
Peptide 2 (VFLVLLPLV): 1 results
Peptide 3 (FLVLLPLVS): 1 results
Peptide 4 (LVLLPLVSS): 1 results
Peptide 5 (VLLPLVSSQ): 1 results
Peptide 6 (LLPLVSSQC): 1 results
Peptide 7 (LPLVSSQCV): 1 results
Peptide 8 (PLVSSQCVN): 1 results
Peptide 9 (LVSSQCVNL): 1 results
Peptide 10 (VSSQCVNLT): 1 results
Peptide 11 (SSQCVNLTT): 1 results
Peptide 12 (SQCVNLTTR): 1 results
Peptide 13 (QCVNLTTRT): 1 results
Peptide 14 (CVNLTTRTQ): 1 results
Peptide 15 (VNLTTRTQL): 1 results
Peptide 16 (NLTTRTQLP): 1 results
Peptide 17 (LTTRTQLPP): 1 results
Peptide 18 (TTRTQLPPA): 1 results
Peptide 19 (TRTQLPPAY): 1 results
Peptide 20 (RTQLPPAYT): 1 results
Peptide 21 (TQLPPAYTN): 1 results
Peptide 22 (QLPPAYTNS): 1 results
Peptide 23 (LPPAYTNSF): 1 results
Peptide 24 (PPAYTNSFT): 1 results
Peptide 25 (PAYTNSFTR): 1 results
Peptide 26 (AYTNSFTRG): 1 results
Peptide 27 (YTNSFTRGV): 1 results
Peptide 28 (TNSFTRGVY): 1 results
Peptide 29 (NSFTRGVYY): 

T-cell epitope prediction

In [ ]:
tcell_results = []

for i, pep in enumerate(peptides[:50]):
    try:
        if len(pep) != 9:
            continue

        tcell_res = iedb.query_tcell_epitope(
            method="smm",
            sequence=pep,
            allele="HLA-A*02:01",
            length=9,
            proteasome='immuno'
        )

        if isinstance(tcell_res, pd.DataFrame):
            print(f"Peptide {i} ({pep}): {len(tcell_res)} T-cell results")
            tcell_res['peptide'] = pep
            tcell_results.append(tcell_res)

        time.sleep(0.5)

    except Exception as e:
        print(f"Error: {str(e)}")
        continue

if tcell_results:
    tcell_df = pd.concat(tcell_results, ignore_index=True)
    print(f"\n📊 T-cell DataFrame shape: {tcell_df.shape}")
    print(tcell_df.head())

Peptide 0 (MFVFLVLLP): 1 T-cell results
Peptide 1 (FVFLVLLPL): 1 T-cell results
Peptide 2 (VFLVLLPLV): 1 T-cell results
Peptide 3 (FLVLLPLVS): 1 T-cell results
Peptide 4 (LVLLPLVSS): 1 T-cell results
Peptide 5 (VLLPLVSSQ): 1 T-cell results
Peptide 6 (LLPLVSSQC): 1 T-cell results
Peptide 7 (LPLVSSQCV): 1 T-cell results
Peptide 8 (PLVSSQCVN): 1 T-cell results
Peptide 9 (LVSSQCVNL): 1 T-cell results
Peptide 10 (VSSQCVNLT): 1 T-cell results
Peptide 11 (SSQCVNLTT): 1 T-cell results
Peptide 12 (SQCVNLTTR): 1 T-cell results
Peptide 13 (QCVNLTTRT): 1 T-cell results
Peptide 14 (CVNLTTRTQ): 1 T-cell results
Peptide 15 (VNLTTRTQL): 1 T-cell results
Peptide 16 (NLTTRTQLP): 1 T-cell results
Peptide 17 (LTTRTQLPP): 1 T-cell results
Peptide 18 (TTRTQLPPA): 1 T-cell results
Peptide 19 (TRTQLPPAY): 1 T-cell results
Peptide 20 (RTQLPPAYT): 1 T-cell results
Peptide 21 (TQLPPAYTN): 1 T-cell results
Peptide 22 (QLPPAYTNS): 1 T-cell results
Peptide 23 (LPPAYTNSF): 1 T-cell results
Peptide 24 (PPAYTNSFT): 1 

Check T-Cell Dataframe columns

In [ ]:
print("T-cell DataFrame columns:")
print(tcell_df.columns.tolist())
print("\nFirst few rows:")
print(tcell_df.head())

print("\n\nMHC DataFrame columns:")
print(df.columns.tolist())
print("\nFirst few rows:")
print(df.head())

T-cell DataFrame columns:
['allele', 'seq_num', 'start', 'end', 'length', 'peptide', 'proteasome_score', 'tap_score', 'mhc_score', 'processing_score', 'total_score', 'ic50_score']

First few rows:
        allele seq_num start end length    peptide proteasome_score tap_score  \
0  HLA-A*02:01       1     1   9      9  MFVFLVLLP           0.5934    0.1046   
1  HLA-A*02:01       1     1   9      9  FVFLVLLPL           1.1911    0.4330   
2  HLA-A*02:01       1     1   9      9  VFLVLLPLV           0.9702    0.1747   
3  HLA-A*02:01       1     1   9      9  FLVLLPLVS           1.1945   -1.0006   
4  HLA-A*02:01       1     1   9      9  LVLLPLVSS           1.1459   -0.9343   

  mhc_score processing_score total_score  ic50_score  
0   -4.4021           0.6979     -3.7042  25241.2001  
1   -1.4051           1.6242      0.2190     25.4162  
2   -2.4701           1.1448     -1.3253    295.1957  
3   -2.8661           0.1939     -2.6722    734.6999  
4   -4.0931           0.2117     -3.8815 

## **Merge Both Dataset**

In [ ]:
combined_df = df.merge(tcell_df,
                       on='peptide',
                       how='inner',
                       suffixes=('_mhc', '_tcell'))

print(f"Combined DataFrame shape: {combined_df.shape}")
print(f"Columns: {combined_df.columns.tolist()}")
print(combined_df.head())

combined_df.to_csv("epitope_prediction_combined.csv", index=False)

Combined DataFrame shape: (50, 21)
Columns: ['allele_mhc', 'seq_num_mhc', 'start_mhc', 'end_mhc', 'length_mhc', 'peptide', 'core', 'icore', 'score', 'percentile_rank', 'allele_tcell', 'seq_num_tcell', 'start_tcell', 'end_tcell', 'length_tcell', 'proteasome_score', 'tap_score', 'mhc_score', 'processing_score', 'total_score', 'ic50_score']
    allele_mhc seq_num_mhc start_mhc end_mhc length_mhc    peptide       core  \
0  HLA-A*02:01           1         1       9          9  MFVFLVLLP  MFVFLVLLP   
1  HLA-A*02:01           1         1       9          9  FVFLVLLPL  FVFLVLLPL   
2  HLA-A*02:01           1         1       9          9  VFLVLLPLV  VFLVLLPLV   
3  HLA-A*02:01           1         1       9          9  FLVLLPLVS  FLVLLPLVS   
4  HLA-A*02:01           1         1       9          9  LVLLPLVSS  LVLLPLVSS   

       icore    score percentile_rank  ... seq_num_tcell start_tcell  \
0  MFVFLVLLP  0.00013              31  ...             1           1   
1  FVFLVLLPL   0.0469        

Alternative code for a more selective columns

In [ ]:
mhc_cols = df[['peptide', 'allele', 'score', 'percentile_rank']]
tcell_cols = tcell_df[['peptide']].copy()

print(tcell_df.columns.tolist())

combined_df = mhc_cols.merge(tcell_cols, on='peptide', how='inner')
print(combined_df.head())

['allele', 'seq_num', 'start', 'end', 'length', 'peptide', 'proteasome_score', 'tap_score', 'mhc_score', 'processing_score', 'total_score', 'ic50_score']
     peptide       allele    score percentile_rank
0  MFVFLVLLP  HLA-A*02:01  0.00013              31
1  FVFLVLLPL  HLA-A*02:01   0.0469             2.2
2  VFLVLLPLV  HLA-A*02:01  0.00903             5.2
3  FLVLLPLVS  HLA-A*02:01   0.0137             4.3
4  LVLLPLVSS  HLA-A*02:01  0.00392             7.7


**top epitope (MFVFLVLLP) shows:**

IC50: 0.00013 nM → Excellent MHC binder
Percentile: 31 → Good candidate
This peptide will likely bind strongly to HLA-A*02:01

Data Collection

In [90]:
# Save to current directory (Colab's /content)
combined_df.to_csv("epitope_prediction_combined.csv", index=False)

# In Colab, files are already there
# When you push to GitHub, include this CSV file